# PCA — RemoveNoise, batch-stratified held-out

- The per-slice mean ("noise") is **shaped on one set of DMSO wells and applied to a held-out
  set** — no in-sample / circular removal. The split is **batch-stratified** (P1 / P2 from
  `layout_id`): wells are split 50/50 within each batch, the per-(batch, slice) mean is
  estimated on the estimate wells, and the corrected held-out wells are pooled.
- `before` = per-plate-normalized DMSO (slice mean still present).
  `after`  = the same held-out wells minus the independently-estimated per-slice mean.
- Metrics are reported for the **top PCs** (PC1–PC5), not only PC2, and each slice effect gets
  a **permutation p-value** (shuffle slice labels) alongside the ANOVA F-test.

In [ ]:
# --- repo path bootstrap (added by the port) ---
import sys, pathlib
ROOT = next(p for p in pathlib.Path.cwd().parents if (p / "utils" / "paths.py").is_file())
sys.path.insert(0, str(ROOT))
from utils.paths import (profiles, features, feature_output, figdir, metadata,
                         data_dir, external, require)
from utils.panels import save_panel

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.text import Text
import seaborn as sns; sns.set_style("white")
import matplotlib as mpl
from sklearn.decomposition import PCA

from scipy.stats import pearsonr, f_oneway

%matplotlib inline
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42


# Set current working directory

cell_line = "HCT116"

# --- analysis settings ---
N_PCS  = 5       # report r / eta2 / p for PC1..PC_N_PCS
N_PERM = 1000    # permutations for the slice-effect p-value
SEED   = 0       # reproducible batch-stratified split & permutations

In [ ]:
def pDose(x):
    '''This is just a helper function, to compute easily log transformed concentrations used in drug discovery'''
    return(-np.log10(1e-6*x))
def inverse_pDose(y):
    '''Inverse function of pDose'''
    return 10**(-y) / 1e-6
def list_features(df):
    # List features
    list_of_selected_features = list(df.columns.values)
    list_of_metadata = list(df.columns[df.columns.str.contains("Metadata_")])
    list_of_selected_features = list(set(list_of_selected_features) - set(list_of_metadata))
    return list_of_selected_features

In [ ]:
# --- slice-effect metric helpers ---
def eta2_1d(x, site):
    """Fraction of variance in a single axis explained by slice (one-way ANOVA between/total SS)."""
    x = np.asarray(x, float)
    grand = x.mean()
    total = ((x - grand) ** 2).sum()
    between = sum((site == s).sum() * (x[site == s].mean() - grand) ** 2 for s in np.unique(site))
    return between / total if total > 0 else np.nan

def global_eta2(mat, site):
    """Fraction of TOTAL feature variance explained by slice (basis-independent)."""
    mat = np.asarray(mat, float)
    grand = mat.mean(0)
    total = ((mat - grand) ** 2).sum(0)
    between = np.zeros(mat.shape[1])
    for s in np.unique(site):
        m = site == s
        between += m.sum() * (mat[m].mean(0) - grand) ** 2
    return between.sum() / total.sum()

def perm_p(stat_func, data_arg, site, observed, n_perm, seed):
    """Permutation p-value: p = (1 + #{eta2(shuffled) >= observed}) / (n_perm + 1)."""
    rng = np.random.default_rng(seed)
    count = 0
    for _ in range(n_perm):
        count += stat_func(data_arg, rng.permutation(site)) >= observed
    return (count + 1) / (n_perm + 1)

In [ ]:
# Save the data
ImagesOut = str(figdir('Fig2')) + '/'

if not os.path.exists(ImagesOut): 
        os.makedirs(ImagesOut)

In [ ]:
# Set up the plotting parameters
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
dpi = 300
figformat = 'pdf'

In [ ]:
# Load the data
file = profiles("exp1_main", "normalized_data_merged_HCT116.csv")
data = pd.read_csv(file)
# "before" state: per-plate normalized DMSO (plate offset removed, slice mean still present)
data = data.query('Metadata_name == "dmso" and Metadata_normalization == "per_plate"').copy()

# Attach acquisition batch (P1 / P2) from layout_id
meta = pd.read_csv(metadata("spher_colo52-metadata.csv", "exp1_main"))
meta['Metadata_batch'] = meta['layout_id'].str.extract(r'-(P\d)-')
bc2batch = meta.drop_duplicates('barcode').set_index('barcode')['Metadata_batch']
data['Metadata_batch'] = data['Metadata_Barcode'].astype(str).map(bc2batch)

cmap = sns.color_palette("YlOrRd",
                         n_colors=len(data['Metadata_Site'].unique()))

print(data.groupby('Metadata_batch')['Metadata_Barcode'].unique())
print(data.groupby('Metadata_batch')['Metadata_PlateWell'].nunique())

## Batch-stratified held-out correction

Split wells 50/50 **within each batch**; shape the per-(batch, slice) mean on the estimate wells, subtract it from the pooled held-out wells to form `after`.

In [ ]:
SITE, WELL, BATCH = 'Metadata_Site', 'Metadata_PlateWell', 'Metadata_batch'
features = list_features(data)
rng = np.random.default_rng(SEED)

# Batch-stratified split: 50/50 of the wells within each batch
est_wells, apply_wells = set(), set()
for b in sorted(data[BATCH].dropna().unique()):
    # sorted(): .unique() follows row order, so without this the split depends on
    # how the input table happens to be ordered rather than on SEED alone.
    w = np.array(sorted(data.loc[data[BATCH] == b, WELL].unique()), dtype=object)
    rng.shuffle(w)
    k = len(w) // 2
    est_wells   |= set(w[:k])
    apply_wells |= set(w[k:])
assert est_wells.isdisjoint(apply_wells), "well leakage between estimate and apply!"

est    = data[data[WELL].isin(est_wells)]
apply_ = data[data[WELL].isin(apply_wells)]

# Shape the per-(batch, slice) mean on the estimate wells, apply to the held-out wells
mean_per_slice = est.groupby([BATCH, SITE])[features].mean()
keys       = list(zip(apply_[BATCH].values, apply_[SITE].values))
before_mat = apply_[features].values
after_mat  = before_mat - mean_per_slice.loc[keys].values

slice_info = apply_[SITE].values           # slice label of each held-out row
states     = ['before', 'after']
mats       = {'before': before_mat, 'after': after_mat}

print(f"estimate wells: {len(est_wells)}   held-out (apply) wells: {len(apply_wells)}")
print(f"held-out rows:  {len(apply_)}   features: {len(features)}")

In [ ]:
# Calculate the PCA embedding
# Fit the PCA basis on the "before" held-out matrix; project "after" into the SAME basis
# so the axes are directly comparable across before/after.
pca = PCA(n_components=N_PCS)
embedding = {'before': pca.fit_transform(before_mat),
             'after':  pca.transform(after_mat)}

variance_explained = pca.explained_variance_ratio_ * 100  # Convert to percentage

In [ ]:
# Shared axis limits across before/after so the two scatters are directly comparable
_all = np.vstack([embedding['before'][:, :2], embedding['after'][:, :2]])
_pad = 0.05 * (_all.max(0) - _all.min(0))
xlim = (_all[:, 0].min() - _pad[0], _all[:, 0].max() + _pad[0])
ylim = (_all[:, 1].min() - _pad[1], _all[:, 1].max() + _pad[1])

for state in states:

    embedding1 = embedding[state]

    ## PCA - slice colouring
    fig = plt.figure(figsize=(6, 3))
    ax = sns.scatterplot(
        x=embedding1[:, 0],
        y=embedding1[:, 1],
        hue=slice_info,
        alpha=(0.7),
        marker="o",
        palette=cmap,
        s=30,
        edgecolor='none',
    )
    plt.legend(bbox_to_anchor=(1.3, 1), loc='upper right')

    # Add variance explained to axis labels
    plt.xlabel(f'PC1 ({variance_explained[0]:.2f}%)', size=18)
    plt.ylabel(f'PC2 ({variance_explained[1]:.2f}%)', size=18)

    # Constant axis limits for both plots; reversed y keeps the inverted-axis look
    plt.xlim(xlim)
    plt.ylim(ylim[1], ylim[0])

    # PC2 slice-effect stats in the title: eta2, r and its p-value
    pc2 = embedding1[:, 1]
    r2, p2 = pearsonr(pc2, slice_info)
    e2 = eta2_1d(pc2, slice_info)
    plt.title(f'{state} - {cell_line}\n'
              rf'PC2: $\eta^2$={e2:.2f}, r={r2:+.2f}, p={p2:.1e}', size=11)
    plt.show()

    # embedding1 is a bare 2-D ndarray, which _write_table cannot accept; hand it
    # the values actually plotted (x, y and the hue) instead.
    save_panel(fig, f'Fig2g_{state}',
           data=pd.DataFrame({'PC1': embedding1[:, 0],
                              'PC2': embedding1[:, 1],
                              'slice': slice_info}),
           caption=f'PCA {state} batch stratification, {cell_line}',
           notebook='analysis/3_Figure2/RemoveNoise/5_PCA_RemoveNoise_BatchStratified.ipynb')